In [0]:
%sql

CREATE OR REPLACE TABLE gold.fact_sales
USING DELTA
AS

SELECT

    s.order_number,

    s.customer_key,

    s.product_key,

    s.order_date,

    s.shipping_date,

    s.due_date,

    s.days_to_ship,

    s.quantity,

    s.price,

    s.sales_amount,

    p.cost,

    (s.quantity * p.cost) AS total_cost,

    (s.sales_amount - (s.quantity * p.cost)) AS profit,

    ROUND(

        ((s.sales_amount - (s.quantity * p.cost))
        / NULLIF(s.sales_amount,0))*100,

        2

    ) AS profit_margin,

    CASE

        WHEN s.days_to_ship <= 3 THEN 'Fast'

        WHEN s.days_to_ship <= 7 THEN 'Normal'

        ELSE 'Delayed'

    END AS shipping_performance,

    s.order_year,

    s.order_month

FROM silver.sales s

LEFT JOIN silver.products p

ON s.product_key = p.product_key;

In [0]:
%sql
SELECT *
FROM gold.fact_sales 
LIMIT 20;

In [0]:
%sql
SELECT
    MIN(days_to_ship) AS min_days,
    MAX(days_to_ship) AS max_days,
    COUNT(*) AS total_orders,
    COUNT(days_to_ship) AS populated_days,
    SUM(CASE WHEN days_to_ship IS NULL THEN 1 ELSE 0 END) AS blank_days,
    SUM(CASE WHEN days_to_ship < 0 THEN 1 ELSE 0 END) AS negative_days
FROM gold.fact_sales;

In [0]:
%sql
SELECT
    days_to_ship,
    COUNT(*) AS records
FROM gold.fact_sales
GROUP BY days_to_ship
ORDER BY days_to_ship;